# MoMo-FDVS P11 — governed structured-model training

This is the authorised reportable training surface for P11. It checks out one immutable code revision, validates the group-isolated controlled dataset, fits only on the training partition, selects thresholds only on validation, evaluates the held-out test partition once, and exports provenance evidence. The data is controlled/synthetic only; the results cannot support provider-wide or production claims.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
TRAINING_COMMIT_SHA = "a914f065070558b5b601e6f49cf1691ff7bf9d42"
WORKSPACE = Path("/content/momo-fraud-detection")
assert sys.version_info[:2] == (3, 12), sys.version
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, str(WORKSPACE)], check=True)
subprocess.run(["git", "checkout", "--quiet", TRAINING_COMMIT_SHA], cwd=WORKSPACE, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKSPACE, text=True).strip()
assert head == TRAINING_COMMIT_SHA, (head, TRAINING_COMMIT_SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--requirement", "ml/requirements-dev.lock"], cwd=WORKSPACE, check=True)
print({"training_commit_sha": head, "python": sys.version})

In [ ]:
import os

environment = os.environ.copy()
environment["PYTHONPATH"] = str(WORKSPACE / "ml" / "src")
subprocess.run([sys.executable, "scripts/verify_ml.py"], cwd=WORKSPACE, env=environment, check=True)
print("P11 Colab preflight passed; training has not started in this cell.")

In [ ]:
OUTPUT_DIR = Path("/content/p11-structured-output")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
command = [
    sys.executable, "-m", "momo_fdvs_ml", "train-structured",
    "--dataset", str(WORKSPACE / "ml/data/controlled/structured_features.csv"),
    "--source-manifest", str(WORKSPACE / "ml/data/controlled/manifest.csv"),
    "--output-dir", str(OUTPUT_DIR),
    "--model-version", "structured-rf-controlled-v1",
    "--training-commit-sha", TRAINING_COMMIT_SHA,
]
subprocess.run(command, cwd=WORKSPACE, env=environment, check=True)

In [ ]:
import hashlib
import json

report = json.loads((OUTPUT_DIR / "structured_evaluation_report.json").read_text())
assert report["training_commit_sha"] == TRAINING_COMMIT_SHA
assert report["dataset_scope"] == "controlled_synthetic_only"
assert report["structured_dataset_hash"] == "30a74b15fe34ef229edd7b28d25b334add7d79e2a73d06d9baae3ba560dda07f"
assert report["held_out_test"]["source_group_count"] == 1
artifact = OUTPUT_DIR / "structured-rf-controlled-v1.joblib"
artifact_sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
assert artifact_sha == report["artifact"]["sha256"]
print(json.dumps({
    "training_commit_sha": report["training_commit_sha"],
    "dataset_scope": report["dataset_scope"],
    "structured_dataset_hash": report["structured_dataset_hash"],
    "feature_schema_hash": report["feature_schema_hash"],
    "artifact_sha256": artifact_sha,
    "held_out_test": report["held_out_test"],
    "acceptance_passed": report["acceptance_passed"],
    "limitations": report["limitations"],
}, indent=2, sort_keys=True))

In [ ]:
subprocess.run([
    sys.executable, "-m", "momo_fdvs_ml", "verify-structured-artifact",
    "--artifact", str(artifact),
    "--sha256", artifact_sha,
    "--schema-hash", report["feature_schema_hash"],
], cwd=WORKSPACE, env=environment, check=True)
print("Independent artifact hash/schema verification passed.")

In [ ]:
import zipfile
from google.colab import files

safe_evidence = Path("/content/P11_SAFE_EVIDENCE.zip")
with zipfile.ZipFile(safe_evidence, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ["structured_evaluation_report.json", "STRUCTURED_MODEL_CARD.md", "structured_registry_payload.json", "structured_confusion_matrix.png"]:
        archive.write(OUTPUT_DIR / filename, arcname=filename)
print({"safe_evidence": str(safe_evidence), "private_artifact": str(artifact), "artifact_sha256": artifact_sha})
files.download(str(safe_evidence))
files.download(str(artifact))